# Simulation Visualization

Final model selection and visualization: global epistasis landscape,
predicted vs measured phenotypes, enrichment comparisons, and shift
correlation plots for the chosen lasso strength.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import os
import pickle
import itertools

import pandas as pd
import numpy as np
from plotnine import *
import multidms
import multidms.plot

from notebooks._common import load_config, build_phenotype_fxn_dict, reconstruct_simulators

In [ ]:
cfg = load_config(config_path, profile)
sim = cfg["simulation"]

seed = cfg["seed"]
output_dir = sim["output_dir"]
lasso_choice = sim["lasso_choice"]
wt_latent = sim["wt_latent"]
sigmoid_phenotype_scale = sim["sigmoid_phenotype_scale"]

warnings.simplefilter("ignore")
%matplotlib inline

## Load data

In [ ]:
fit_collection_df = pickle.load(open(f"{output_dir}/fit_collection.pkl", "rb"))
mut_effects_df = pd.read_csv(f"{output_dir}/simulated_muteffects.csv")
model_vs_truth = pd.read_csv(f"{output_dir}/model_vs_truth_beta_shift.csv")
sparsity = pd.read_csv(f"{output_dir}/fit_sparsity.csv")
replicate_corr = pd.read_csv(f"{output_dir}/library_replicate_correlation.csv")
phenotype = pd.read_csv(f"{output_dir}/model_vs_truth_variant_phenotype.csv")
cv_loss = pd.read_csv(f"{output_dir}/cross_validation_loss.csv")

model_collection = multidms.model_collection.ModelCollection(fit_collection_df)
print(f"Loaded all evaluation data. Lasso choice: {lasso_choice}")

## Global epistasis landscape

Visualize the sigmoidal global epistasis function for the chosen model.

In [ ]:
multidms.plot.ge_landscape(
    model_collection.fit_models
    .query(f"fusionreg == {lasso_choice} and measurement_type == 'loose_bottle' and library == 'lib_1'")
    .iloc[0]
    .model
)

## Reconstruct phenotype functions for ground truth comparison

In [ ]:
phenotype_fxn_dict_h1, phenotype_fxn_dict_h2 = reconstruct_simulators(
    sim, mut_effects_df, seed
)
print("Phenotype functions reconstructed")

## Build variants dataframe with ground truth

In [ ]:
variants_df = pd.concat([
    row.model.get_variants_df(phenotype_as_effect=False)
    .assign(library=row.library, measurement_type=row.measurement_type, fusionreg=row.fusionreg)
    .rename({
        "predicted_func_score": "predicted_phenotype",
        "predicted_latent": "predicted_latent_phenotype",
        "func_score": "measured_phenotype",
    }, axis=1)
    .assign(
        predicted_enrichment=lambda x: 2 ** x["predicted_phenotype"],
        measured_enrichment=lambda x: 2 ** x["measured_phenotype"],
        fit_idx=idx,
    )
    for idx, row in fit_collection_df.iterrows()
])

variants_df = pd.concat([
    variants_df.query("condition == @homolog").assign(
        true_latent_phenotype=lambda x: x["aa_substitutions"].apply(pfxn["latentPhenotype"]),
        true_observed_phenotype=lambda x: x["aa_substitutions"].apply(pfxn["observedPhenotype"]),
        true_enrichment=lambda x: x["aa_substitutions"].apply(pfxn["observedEnrichment"]),
    )
    for homolog, pfxn in [("h1", phenotype_fxn_dict_h1), ("h2", phenotype_fxn_dict_h2)]
])

variants_df["measurement_type"] = pd.Categorical(
    variants_df["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)

## Predicted vs measured phenotypes

For the chosen lasso strength, plot predicted latent and observed phenotypes
against measured values.

In [ ]:
data = variants_df.query(f"fusionreg == {lasso_choice}")
for x, y in itertools.combinations(
    ["predicted_latent_phenotype", "predicted_phenotype", "measured_phenotype"], 2
):
    p = (
        ggplot(data.sample(frac=1.0))
        + geom_point(aes(x=x, y=y), alpha=0.05, color="grey")
        + ylab(y)
        + xlab(x)
        + facet_grid("~measurement_type")
        + theme_classic()
        + theme(figure_size=(6.5, 3.5), axis_text_x=element_text(angle=90))
    )
    if x == "predicted_latent_phenotype":
        p += geom_point(aes(x="true_latent_phenotype", y="true_observed_phenotype"), size=0.5)
    else:
        p += geom_abline(slope=1, intercept=0)
    _ = p.draw(show=True)

## Enrichment comparison

Compare predicted and measured enrichments against ground truth.

In [ ]:
p = (
    ggplot(
        data.melt(
            id_vars=["measurement_type", "library", "true_enrichment"],
            value_vars=["predicted_enrichment", "measured_enrichment"],
            var_name="enrichment_type",
            value_name="enrichment",
        ),
        aes("true_enrichment", "enrichment"),
    )
    + geom_abline(slope=1, intercept=0)
    + geom_point(alpha=0.05, size=0.5)
    + facet_grid("enrichment_type~measurement_type", scales="free_y")
    + theme_classic()
    + theme(figure_size=(6.5, 5))
)
_ = p.draw(show=True)

## Shift correlations

Correlation between true and predicted shifts for the chosen lasso strength.

In [ ]:
# Rebuild collection_muts_df for shift correlation plot
groupby = ("library", "measurement_type", "fusionreg")
collection_muts_df = (
    model_collection.split_apply_combine_muts(groupby=groupby)
    .reset_index()
    .rename({"beta_h1": "predicted_beta", "shift_h2": "predicted_shift_h2"}, axis=1)
    .merge(
        mut_effects_df.rename(
            {"beta_h1": "true_beta", "beta_h2": "true_beta_h2", "shift": "true_shift"}, axis=1
        ),
        on="mutation",
    )
)

collection_muts_df["measurement_type"] = pd.Categorical(
    collection_muts_df["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)

for x, y in itertools.combinations(["true_shift", "predicted_shift_h2"], 2):
    p = (
        ggplot(collection_muts_df.query(f"fusionreg == {lasso_choice}"))
        + geom_point(aes(x=x, y=y), alpha=0.5)
        + geom_abline(slope=1, intercept=0)
        + ylab(y)
        + xlab(x)
        + facet_grid("library~measurement_type")
        + theme_classic()
        + theme(figure_size=(6.5, 5), axis_text_x=element_text(angle=90))
    )
    _ = p.draw(show=True)